In [5]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

import nltk
# Download NLTK data once (stopwords, wordnet for lemmatization)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer

# --- Data paths (relative to notebook in model/) ---
DATA_DIR = Path("../10_categories_of_yahoo_answers_for_nlp_tasks_csv")
TRAIN_PATH = DATA_DIR / "train_mini.csv"
TEST_PATH = DATA_DIR / "test.csv"



In [6]:
STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()

def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def preprocess_df(df):
    title = df["question_title"].fillna("").astype(str)
    content = df["question_content"].fillna("").astype(str)
    question = (title + " " + content).str.strip()
    return question.apply(normalize_text)

def tokenize_for_vectorizer(text):
    text = text.lower().strip()
    tokens = re.findall(r"\b\w+\b", text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    return tokens


# Load train and test data
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# Preprocess text and store in new column 
train_df["text"] = preprocess_df(train_df)
test_df["text"] = preprocess_df(test_df)

# Target: class_index
train_df["class_index"] = pd.to_numeric(train_df["class_index"], errors="coerce").astype("Int64")
test_df["class_index"] = pd.to_numeric(test_df["class_index"], errors="coerce").astype("Int64")

# Clean:Drop rows with missing class 
train_df = train_df.dropna(subset=["class_index"])
test_df = test_df.dropna(subset=["class_index"])

# Create train/validation split from train_df
X_raw = train_df["text"].values
y = train_df["class_index"].astype(int).values

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_raw,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# Test set remains the provided test.csv
X_test_raw = test_df["text"].values
y_test = test_df["class_index"].astype(int).values

print("Train size:", len(y_train), "Val size:", len(y_val), "Test size:", len(y_test))
print("Class distribution (train):\n", pd.Series(y_train).value_counts().sort_index())
print("----------")
print("Sample cleaned text (train):")
print(X_train_raw[:3])

Train size: 200000 Test size: 59999
Class distribution (train):
 1     20000
2     20000
3     20000
4     20000
5     20000
6     20000
7     20000
8     20000
9     20000
10    20000
Name: count, dtype: int64
----------
Sample cleaned text:
<StringArray>
[                          'where is there a record of all the women who were 100 meter backstroke gold medal winners in the olympic games',
 'this is getting hard. can u help me? i gone through it and can't find the answer.? what does breeding mean? i need help. help help help.',
                                                  'does anybody just want to say anything, just get something off there chest or watever ?']
Length: 3, dtype: str


In [7]:
# Vectorize text using CountVectorizer on train, then transform val and test
count_vectorizer = CountVectorizer(
    tokenizer=tokenize_for_vectorizer,
    lowercase=False,
    max_features=10000,
)

X_train = count_vectorizer.fit_transform(X_train_raw)
X_val = count_vectorizer.transform(X_val_raw)
X_test = count_vectorizer.transform(X_test_raw)
print("Feature matrix shapes - train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)

# Resulting feature matrix (sparse)
#print(X_train[4])
#print("Sample tokens (first train doc):", tokenize_for_vectorizer(X_train_raw[4])[:25])

/Users/tyleryue/Desktop/4NL3/project/project-4nl3/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Feature matrix shape train: (200000, 10000) test: (59999, 10000)


In [10]:
# Random Baseline Model
rng = np.random.default_rng(1)
classes = np.unique(y_train)
y_pred_random = rng.choice(classes, size=len(y_test))
acc_random = accuracy_score(y_test, y_pred_random)
print("Random baseline accuracy:", round(acc_random, 4))
print(classification_report(y_test, y_pred_random, zero_division=0))

Random baseline accuracy: 0.1017
              precision    recall  f1-score   support

           1       0.10      0.10      0.10      6000
           2       0.10      0.10      0.10      6000
           3       0.10      0.11      0.11      6000
           4       0.10      0.10      0.10      6000
           5       0.10      0.10      0.10      6000
           6       0.10      0.10      0.10      6000
           7       0.10      0.10      0.10      6000
           8       0.10      0.10      0.10      6000
           9       0.11      0.10      0.10      5999
          10       0.10      0.10      0.10      6000

    accuracy                           0.10     59999
   macro avg       0.10      0.10      0.10     59999
weighted avg       0.10      0.10      0.10     59999



In [11]:
# Majority Vote Baseline Model
from collections import Counter
majority_class = Counter(y_train).most_common(1)[0][0]
y_pred_majority = np.full(len(y_test), majority_class)
acc_majority = accuracy_score(y_test, y_pred_majority)
print("Majority baseline accuracy:", round(acc_majority, 4))
print("Predicted class for all:", majority_class)
print(classification_report(y_test, y_pred_majority, zero_division=0))

Majority baseline accuracy: 0.1
Predicted class for all: 6
              precision    recall  f1-score   support

           1       0.00      0.00      0.00      6000
           2       0.00      0.00      0.00      6000
           3       0.00      0.00      0.00      6000
           4       0.00      0.00      0.00      6000
           5       0.00      0.00      0.00      6000
           6       0.10      1.00      0.18      6000
           7       0.00      0.00      0.00      6000
           8       0.00      0.00      0.00      6000
           9       0.00      0.00      0.00      5999
          10       0.00      0.00      0.00      6000

    accuracy                           0.10     59999
   macro avg       0.01      0.10      0.02     59999
weighted avg       0.01      0.10      0.02     59999



In [12]:
# Logistical Regression Baseline Model 
lr = LogisticRegression(max_iter=500, C=1.0, solver="lbfgs", random_state=1)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
print("Logistic Regression accuracy:", round(acc_lr, 4))
print(classification_report(y_test, y_pred_lr, zero_division=0))

Logistic Regression accuracy: 0.6532
              precision    recall  f1-score   support

           1       0.57      0.51      0.54      6000
           2       0.58      0.69      0.63      6000
           3       0.70      0.73      0.71      6000
           4       0.49      0.48      0.49      6000
           5       0.80      0.81      0.81      6000
           6       0.83      0.81      0.82      6000
           7       0.49      0.47      0.48      6000
           8       0.66      0.63      0.65      6000
           9       0.68      0.71      0.70      5999
          10       0.73      0.68      0.70      6000

    accuracy                           0.65     59999
   macro avg       0.65      0.65      0.65     59999
weighted avg       0.65      0.65      0.65     59999



In [14]:
# Random Forest Baseline Model
rf = RandomForestClassifier(n_estimators=100, max_depth=50, random_state=1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print("Random Forest accuracy:", round(acc_rf, 4))
print(classification_report(y_test, y_pred_rf, zero_division=0))

Random Forest accuracy: 0.522
              precision    recall  f1-score   support

           1       0.60      0.38      0.47      6000
           2       0.24      0.76      0.37      6000
           3       0.65      0.51      0.57      6000
           4       0.58      0.29      0.39      6000
           5       0.66      0.73      0.69      6000
           6       0.78      0.57      0.66      6000
           7       0.61      0.29      0.40      6000
           8       0.73      0.41      0.53      6000
           9       0.56      0.71      0.63      5999
          10       0.69      0.55      0.61      6000

    accuracy                           0.52     59999
   macro avg       0.61      0.52      0.53     59999
weighted avg       0.61      0.52      0.53     59999

